In [ ]:



# =============================================================================
# 4. MODEL ARCHITECTURE DEFINITION
# =============================================================================
class MultiHeadCrossAttention(layers.Layer):
    def __init__(self, d_model=512, num_heads=8, **kwargs):
        super().__init__(**kwargs)
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.query_dense = layers.Dense(d_model)
        self.key_dense = layers.Dense(d_model)
        self.value_dense = layers.Dense(d_model)
        self.combine_heads = layers.Dense(d_model)
        self.layernorm = layers.LayerNormalization()
        self.add = layers.Add()

    def attention(self, query, key, value):
        score = tf.matmul(query, key, transpose_b=True)
        dim_key = tf.cast(tf.shape(key)[-1], tf.float32)
        scaled_score = score / tf.math.sqrt(dim_key)
        weights = tf.nn.softmax(scaled_score, axis=-1)
        output = tf.matmul(weights, value)
        return output

    def separate_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.num_heads, self.head_dim))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, query_input, key_input, value_input):
        batch_size = tf.shape(query_input)[0]
        query = self.query_dense(query_input)
        key = self.key_dense(key_input)
        value = self.value_dense(value_input)
        
        query = self.separate_heads(query, batch_size)
        key = self.separate_heads(key, batch_size)
        value = self.separate_heads(value, batch_size)
        
        attention_output = self.attention(query, key, value)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (batch_size, -1, self.d_model))
        
        combined = self.combine_heads(concat_attention)
        # Add & Norm
        output = self.layernorm(self.add([query_input, combined]))
        return output

def create_model():
    # --- Define Inputs ---
    image_input = layers.Input(shape=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), name="image_input")
    text_ids_input = layers.Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_ids_input")
    text_mask_input = layers.Input(shape=(CONFIG.MAX_TEXT_LEN,), dtype=tf.int32, name="text_mask_input")
    engineered_input = layers.Input(shape=(engineered_train_feats.shape[1],), name="engineered_input")
    
    # --- Load Pre-trained CLIP Model ---
    clip_model = TFAutoModel.from_pretrained(CONFIG.CLIP_MODEL_NAME)
    
    # Freeze CLIP layers
    clip_model.text_model.trainable = False
    clip_model.vision_model.trainable = False
    
    # --- Get Embeddings ---
    text_embeds = clip_model.text_model(input_ids=text_ids_input, attention_mask=text_mask_input).pooler_output
    image_embeds = clip_model.vision_model(pixel_values=image_input).pooler_output
    
    # Add a sequence dimension for attention
    text_embeds_seq = layers.Reshape((1, -1))(text_embeds)
    image_embeds_seq = layers.Reshape((1, -1))(image_embeds)

    # --- Multi-Head Cross-Modal Attention Block ---
    attention_layer = MultiHeadCrossAttention(d_model=text_embeds.shape[-1], num_heads=8)
    text_to_image_features = attention_layer(text_embeds_seq, image_embeds_seq, image_embeds_seq)
    image_to_text_features = attention_layer(image_embeds_seq, text_embeds_seq, text_embeds_seq)
    
    fused_features = layers.Concatenate()([
        layers.Flatten()(text_to_image_features),
        layers.Flatten()(image_to_text_features)
    ])
    
    # --- Final Fusion and Regression Head ---
    all_features = layers.Concatenate()([fused_features, engineered_input])
    
    x = layers.BatchNormalization()(all_features)
    x = layers.Dense(512, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-5))(x)
    
    output = layers.Dense(1, activation='relu', name='price_output')(x) # ReLU to ensure positive price
    
    model = keras.Model(
        inputs=[image_input, text_ids_input, text_mask_input, engineered_input],
        outputs=output
    )
    
    return model

# Custom SMAPE Metric for monitoring
def smape_metric(y_true, y_pred):
    y_true = tf.expm1(y_true) # Reverse log transform
    y_pred = tf.expm1(y_pred)
    numerator = tf.abs(y_pred - y_true)
    denominator = (tf.abs(y_true) + tf.abs(y_pred)) / 2.0
    return tf.reduce_mean(numerator / (denominator + 1e-8)) * 100.0

# =============================================================================
# 5. TRAINING AND INFERENCE
# =============================================================================
print("5. Starting training with K-Fold Cross-Validation...")
kf = KFold(n_splits=CONFIG.N_SPLITS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(train_df))
test_preds = np.zeros(len(test_df))

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"\n===== FOLD {fold+1}/{CONFIG.N_SPLITS} =====")
    
    # --- Prepare Data for Fold ---
    X_train = {
        'image_input': train_images[train_idx],
        'text_ids_input': train_text_tokens['input_ids'][train_idx],
        'text_mask_input': train_text_tokens['attention_mask'][train_idx],
        'engineered_input': engineered_train_feats[train_idx]
    }
    y_train_fold = y_log[train_idx]
    
    X_val = {
        'image_input': train_images[val_idx],
        'text_ids_input': train_text_tokens['input_ids'][val_idx],
        'text_mask_input': train_text_tokens['attention_mask'][val_idx],
        'engineered_input': engineered_train_feats[val_idx]
    }
    y_val_fold = y_log[val_idx]
    
    # --- Build and Compile Model ---
    keras.backend.clear_session()
    model = create_model()
    optimizer = keras.optimizers.AdamW(learning_rate=CONFIG.LEARNING_RATE)
    # Huber loss is a good proxy for MAE, robust to outliers
    model.compile(optimizer=optimizer, loss='huber', metrics=[smape_metric])
    
    # --- Callbacks ---
    lr_reducer = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1)
    early_stopper = keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1)
    
    # --- Train Model ---
    model.fit(
        X_train, y_train_fold,
        validation_data=(X_val, y_val_fold),
        epochs=CONFIG.EPOCHS,
        batch_size=CONFIG.BATCH_SIZE,
        callbacks=[lr_reducer, early_stopper]
    )
    
    # --- Predict and Store ---
    oof_preds[val_idx] = model.predict(X_val).flatten()
    
    X_test = {
        'image_input': test_images,
        'text_ids_input': test_text_tokens['input_ids'],
        'text_mask_input': test_text_tokens['attention_mask'],
        'engineered_input': engineered_test_feats
    }
    test_preds += model.predict(X_test).flatten() / CONFIG.N_SPLITS
    
    # --- Clean up ---
    del model
    gc.collect()

# =============================================================================
# 6. FINAL EVALUATION AND SUBMISSION
# =============================================================================
print("\nTraining complete.")

# Reverse log transform for final predictions
oof_preds_final = np.expm1(oof_preds)
test_preds_final = np.expm1(test_preds)

# Ensure no negative prices
test_preds_final[test_preds_final < 0] = 0 

# Calculate overall OOF SMAPE
final_oof_smape = smape_metric(y, oof_preds_final).numpy()
print(f"Overall Out-of-Fold SMAPE: {final_oof_smape:.4f}")

# --- Create Submission File ---
submission_df = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_preds_final
})
submission_df.to_csv('submission.csv', index=False)
print("\nSubmission file 'submission.csv' created successfully.")
print("Top 5 predictions:")
print(submission_df.head())

In [ ]:
import os
import gc
import re
import json
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import 
from keras.preprocessing import image
from transformers import AutoTokenizer, TFAutoModel
from sklearn.model_selection import KFold
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm.notebook import tqdm
import requests
from PIL import Image
from io import BytesIO

# Suppress warnings
warnings.filterwarnings("ignore")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

2025-10-12 10:51:15.597019: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760266275.819383      77 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760266275.881707      77 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
# For reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seed(42)

In [7]:
class Config:
    N_SPLITS = 5
    
    OUTPUT_DIR = os.getcwd()
    PREPROCESSED_IMAGE_DIR = os.path.join(OUTPUT_DIR, 'preprocessed_images')
    TRAIN_DIR = "/kaggle/input/amazon-ml-challenge-25/train.csv"
    TEST_DIR = "/kaggle/input/amazon-ml-challenge-25/test.csv"
    IMAGES_TRAIN_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/train"
    IMAGES_TEST_DIR = "/kaggle/input/amazon-ml-images-50-sample/optimized_50_percent_sample/test"
    
    CLIP_MODEL_NAME = 'openai/clip-vit-base-patch32'
    IMG_SIZE = 224
    MAX_TEXT_LEN = 77  # CLIP's max sequence length
    
    BATCH_SIZE = 32
    EPOCHS = 8
    LEARNING_RATE = 1e-4
    
    MAX_FEATURES_TAGS = 500  # Max features for TF-IDF on tags
    MAX_FEATURES_CATS = 200  # Max features for TF-IDF on category candidates

CONFIG = Config()

In [8]:
print("1. Loading data...")
train_df = pd.read_csv(CONFIG.TRAIN_DIR)
test_df = pd.read_csv(CONFIG.TEST_DIR)

1. Loading data...


In [12]:
base_name = lambda file : ".".join(file.split(".")[:-1])
train_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TRAIN_DIR)]
test_samples = [int(base_name(name)) for name in os.listdir(CONFIG.IMAGES_TEST_DIR)]

In [14]:
def sample_data(df, samples) :
    data = []
    for sample in samples :
        data.append(df[df["sample_id"] == sample].values[0])
    return pd.DataFrame(data, columns = df.columns)

train_df = sample_data(train_df, train_samples)
test_df = sample_data(test_df, test_samples)

In [15]:
def extract_features(text: str) -> dict:
    if not isinstance(text, str): text = ""
    features = {}
    PATTERNS = {"item_name": r"Item Name:\s*(.+?)(?:\n|$)", "bullet_points": r"Bullet Point\s*\d*:\s*(.+?)(?=\nBullet Point|$)",}
    for key, pat in PATTERNS.items():
        m = re.findall(pat, text, flags=re.S | re.I)
        if m: features[key] = " ".join([s.strip() for s in m])
    
    # Simple Brand extraction
    brand_match = re.match(r"Item Name:\s*([A-Za-z0-9' -]+)", text)
    if brand_match: features["brand"] = brand_match.group(1).strip()
    else: features["brand"] = "Unknown"

    # Pack count
    pack_match = re.search(r"(?:pack of|set of)\s*(\d+)", text, flags=re.I)
    if pack_match: features["pack_count"] = int(pack_match.group(1))
    else: features["pack_count"] = 1
        
    TAG_PATTERN = r"\b([A-Z][a-z]+(?:[- ][A-Z]?[a-z]+){0,2})\b"
    candidates = re.findall(TAG_PATTERN, text)
    features["tags"] = " ".join(sorted(set([t.strip() for t in candidates if len(t) > 3 and not t.isnumeric()])))
    
    return features

In [17]:
print("2. Applying feature engineering...")
train_features_df = pd.DataFrame([extract_features(text) for text in tqdm(train_df['catalog_content'])])
test_features_df = pd.DataFrame([extract_features(text) for text in tqdm(test_df['catalog_content'])])

2. Applying feature engineering...


  0%|          | 0/37482 [00:00<?, ?it/s]

  0%|          | 0/37499 [00:00<?, ?it/s]

In [19]:
print("3. Vectorizing engineered features...")
# Vectorize 'tags'
tags_vectorizer = TfidfVectorizer(max_features=CONFIG.MAX_FEATURES_TAGS, token_pattern=r'\b[a-zA-Z-]+\b')
train_tags_tfidf = tags_vectorizer.fit_transform(train_features_df['tags'].fillna('')).toarray()
test_tags_tfidf = tags_vectorizer.transform(test_features_df['tags'].fillna('')).toarray()

# Vectorize 'brand' - simple count encoding for this example
brand_counts = train_features_df['brand'].value_counts().to_dict()
train_brand_feat = train_features_df['brand'].map(brand_counts).fillna(1)
test_brand_feat = test_features_df['brand'].map(brand_counts).fillna(1)

# Combine engineered features
engineered_train_feats = np.hstack([
    train_tags_tfidf,
    train_features_df[['pack_count']].fillna(1).values,
    train_brand_feat.values.reshape(-1, 1)
])
engineered_test_feats = np.hstack([
    test_tags_tfidf,
    test_features_df[['pack_count']].fillna(1).values,
    test_brand_feat.values.reshape(-1, 1)
])

# Normalize
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
engineered_train_feats = scaler.fit_transform(engineered_train_feats)
engineered_test_feats = scaler.transform(engineered_test_feats)

print(f"Engineered feature shape: {engineered_train_feats.shape}")


3. Vectorizing engineered features...
Engineered feature shape: (37482, 502)


In [ ]:
print("4. Preparing text and image data...")

# --- Text Tokenization for CLIP ---
tokenizer = AutoTokenizer.from_pretrained(CONFIG.CLIP_MODEL_NAME)
train_text_tokens = tokenizer(text=train_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)
test_text_tokens = tokenizer(text=test_df['catalog_content'].fillna("").tolist(), return_tensors='np', max_length=CONFIG.MAX_TEXT_LEN, padding='max_length', truncation=True)

# --- Image Preprocessing & Caching ---
os.makedirs(CONFIG.PREPROCESSED_IMAGE_DIR, exist_ok=True)
def preprocess_image(sample_id, image_dir):
    img_path = os.path.join(image_dir, f"{sample_id}.jpg")
    filepath = os.path.join(CONFIG.PREPROCESSED_IMAGE_DIR, f"{sample_id}.npy")
    try:
        if os.path.exists(filepath):
            return np.load(filepath)

        img = image.load_img(img_path, target_size=(CONFIG.IMG_SIZE, CONFIG.IMG_SIZE))
        img_array = image.img_to_array(img)
        img_array = np.array(img, dtype=np.float32) / 255.0
        np.save(filepath, img_array)
        return img_array
    except (FileNotFoundError, OSError, ValueError) as e :
        print(f"[WARN] Could not process {sample_id}: {e}")
        return np.zeros((CONFIG.IMG_SIZE, CONFIG.IMG_SIZE, 3), dtype=np.float32)

train_images = np.array([preprocess_image(row.sample_id, row.image_link) for _, row in tqdm(train_df.iterrows(), total=len(train_df))])
test_images = np.array([preprocess_image(row.sample_id, row.image_link) for _, row in tqdm(test_df.iterrows(), total=len(test_df))])

# Target variable preparation
y = train_df['price'].values
y_log = np.log1p(y) # Use log(1+price) for stability
